# One-Step vs Two-Step Model Comparison

This notebook compares the two required project architectures:

- one-step fall detection
- two-step fall detection

The purpose of this notebook is to bring the key outputs together in one place so the team can prepare the analysis section of the report and explain the tradeoffs between both approaches.

## Comparison Goal

The assignment does not only require two different architectures. It also requires a meaningful comparison.

This notebook focuses on:

- collecting available outputs from both pipelines
- organizing them into comparable tables
- identifying strengths and weaknesses
- preparing evidence for the report and final discussion

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()

ONE_STEP_ROOT = PROJECT_ROOT / 'runs' / 'evaluation' / 'one_step_test_eval'
ONE_STEP_SUMMARY_PATH = ONE_STEP_ROOT / 'metrics_summary.json'
ONE_STEP_PER_CLASS_PATH = ONE_STEP_ROOT / 'per_class_metrics.csv'

TWO_STEP_ROOT = PROJECT_ROOT / 'runs' / 'two_step' / 'evaluation'
TWO_STEP_HISTORY_PATH = TWO_STEP_ROOT / 'classifier_history.csv'
TWO_STEP_CROP_PRED_PATH = TWO_STEP_ROOT / 'crop_level_predictions.csv'
TWO_STEP_PIPELINE_PRED_PATH = TWO_STEP_ROOT / 'two_step_predictions.csv'
TWO_STEP_SUMMARY_PATH = TWO_STEP_ROOT / 'two_step_metrics_summary.json'
TWO_STEP_PER_CLASS_PATH = TWO_STEP_ROOT / 'two_step_per_class_metrics.csv'

COMPARISON_ROOT = PROJECT_ROOT / 'runs' / 'comparison'
COMPARISON_ROOT.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['fall detected', 'walk', 'sit']

print('One-step summary path :', ONE_STEP_SUMMARY_PATH)
print('One-step per-class    :', ONE_STEP_PER_CLASS_PATH)
print('Two-step history      :', TWO_STEP_HISTORY_PATH)
print('Two-step crop preds   :', TWO_STEP_CROP_PRED_PATH)
print('Two-step pipeline out :', TWO_STEP_PIPELINE_PRED_PATH)
print('Two-step summary      :', TWO_STEP_SUMMARY_PATH)
print('Two-step per-class    :', TWO_STEP_PER_CLASS_PATH)


## Step 1 - Load the One-Step Evaluation Outputs

The one-step notebook already saves a compact summary and a per-class table. These are the cleanest inputs for comparison.

In [ ]:
assert ONE_STEP_SUMMARY_PATH.exists(), f'Missing file: {ONE_STEP_SUMMARY_PATH}'
assert ONE_STEP_PER_CLASS_PATH.exists(), f'Missing file: {ONE_STEP_PER_CLASS_PATH}'

one_step_summary = json.loads(ONE_STEP_SUMMARY_PATH.read_text(encoding='utf-8'))
one_step_per_class = pd.read_csv(ONE_STEP_PER_CLASS_PATH)

print(one_step_summary)
one_step_per_class


## Step 2 - Load the Two-Step Outputs

The two-step notebook currently provides:

- classifier training history
- crop-level predictions
- end-to-end two-step prediction outputs

At this stage, the crop-level classifier accuracy is the most directly comparable numeric output available from the two-step pipeline notebook.

In [ ]:
assert TWO_STEP_HISTORY_PATH.exists(), f'Missing file: {TWO_STEP_HISTORY_PATH}'
assert TWO_STEP_CROP_PRED_PATH.exists(), f'Missing file: {TWO_STEP_CROP_PRED_PATH}'
assert TWO_STEP_SUMMARY_PATH.exists(), f'Missing file: {TWO_STEP_SUMMARY_PATH}'
assert TWO_STEP_PER_CLASS_PATH.exists(), f'Missing file: {TWO_STEP_PER_CLASS_PATH}'

two_step_history = pd.read_csv(TWO_STEP_HISTORY_PATH)
two_step_crop_preds = pd.read_csv(TWO_STEP_CROP_PRED_PATH)
two_step_pipeline_preds = pd.read_csv(TWO_STEP_PIPELINE_PRED_PATH) if TWO_STEP_PIPELINE_PRED_PATH.exists() else pd.DataFrame()
two_step_summary = json.loads(TWO_STEP_SUMMARY_PATH.read_text(encoding='utf-8'))
two_step_per_class = pd.read_csv(TWO_STEP_PER_CLASS_PATH)

display(two_step_history.tail())
display(two_step_crop_preds.head())
display(two_step_per_class)
print(two_step_summary)

if not two_step_pipeline_preds.empty:
    display(two_step_pipeline_preds.head())


## Step 3 - Summarize the Available Numeric Results

The one-step pipeline provides direct held-out detection metrics from Ultralytics.

The two-step pipeline now provides:

- crop-level classifier accuracy
- end-to-end IoU-based detection-style precision, recall, and F1

This is not identical to YOLO mAP, but it is now much closer to a fair architecture comparison than using classifier accuracy alone.


In [ ]:
one_step_precision = one_step_summary.get('precision')
one_step_recall = one_step_summary.get('recall')
one_step_f1 = one_step_summary.get('f1')
if one_step_f1 is None and one_step_precision is not None and one_step_recall is not None:
    one_step_f1 = (
        2 * one_step_precision * one_step_recall / (one_step_precision + one_step_recall)
        if (one_step_precision + one_step_recall) > 0 else 0.0
    )

selection_metric = 'val_accuracy' if 'val_accuracy' in two_step_history.columns else 'test_accuracy'
best_two_step_row = two_step_history.sort_values(selection_metric, ascending=False).iloc[0]
crop_test_accuracy = two_step_summary.get('crop_test_accuracy')

comparison_summary = pd.DataFrame([
    {
        'model': 'one-step YOLO',
        'precision': one_step_precision,
        'recall': one_step_recall,
        'f1': one_step_f1,
        'mAP50': one_step_summary.get('mAP50'),
        'mAP50-95': one_step_summary.get('mAP50-95'),
        'classifier_selection_metric': None,
        'classifier_selection_accuracy': None,
        'crop_test_accuracy': None,
        'notes': 'Direct held-out object detection benchmark',
    },
    {
        'model': 'two-step detector + classifier',
        'precision': two_step_summary.get('precision'),
        'recall': two_step_summary.get('recall'),
        'f1': two_step_summary.get('f1'),
        'mAP50': None,
        'mAP50-95': None,
        'classifier_selection_metric': selection_metric,
        'classifier_selection_accuracy': float(best_two_step_row[selection_metric]),
        'crop_test_accuracy': float(crop_test_accuracy) if crop_test_accuracy is not None else None,
        'notes': 'End-to-end IoU-based pipeline summary plus held-out crop-level classifier accuracy',
    },
])

comparison_summary


## Step 4 - Visualize One-Step Per-Class Results

The one-step model already has per-class detection metrics, which are useful in the report.

In [ ]:
plot_df = one_step_per_class.melt(
    id_vars=['class_name'],
    value_vars=['precision', 'recall', 'mAP50', 'mAP50-95'],
    var_name='metric',
    value_name='value',
)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x='class_name', y='value', hue='metric')
plt.ylim(0, 1.05)
plt.title('One-Step Baseline Per-Class Performance')
plt.xlabel('Class')
plt.ylabel('Score')
plt.tight_layout()
plt.show()


## Step 5 - Visualize Two-Step Classifier History

The classifier history gives a useful view of whether the second-stage recognizer is learning steadily and whether the held-out crop-level accuracy is stable.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(two_step_history['epoch'], two_step_history['train_accuracy'], label='Train Accuracy')
if 'val_accuracy' in two_step_history.columns:
    plt.plot(two_step_history['epoch'], two_step_history['val_accuracy'], label='Validation Accuracy')
elif 'test_accuracy' in two_step_history.columns:
    plt.plot(two_step_history['epoch'], two_step_history['test_accuracy'], label='Legacy Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Two-Step Classifier Training and Selection Accuracy')
plt.legend()
plt.tight_layout()
plt.show()


## Step 6 - Visualize Two-Step End-to-End Per-Class Results

These metrics show how the full two-step pipeline behaves after combining person detection and crop classification.


In [ ]:
two_step_plot_df = two_step_per_class.melt(
    id_vars=['class_name'],
    value_vars=['precision', 'recall', 'f1'],
    var_name='metric',
    value_name='value',
)
plt.figure(figsize=(9, 5))
sns.barplot(data=two_step_plot_df, x='class_name', y='value', hue='metric')
plt.ylim(0, 1.05)
plt.title('Two-Step End-to-End Per-Class Performance')
plt.xlabel('Class')
plt.ylabel('Score')
plt.tight_layout()
plt.show()


## Step 6 - Export Comparison Tables

Saving the comparison outputs makes them easier to reuse in the report and in later refinement notebooks.

In [ ]:
comparison_summary_path = COMPARISON_ROOT / 'comparison_summary.csv'
one_step_per_class_path = COMPARISON_ROOT / 'one_step_per_class.csv'
two_step_history_path = COMPARISON_ROOT / 'two_step_classifier_history.csv'
two_step_per_class_path = COMPARISON_ROOT / 'two_step_per_class.csv'

comparison_summary.to_csv(comparison_summary_path, index=False)
one_step_per_class.to_csv(one_step_per_class_path, index=False)
two_step_history.to_csv(two_step_history_path, index=False)
two_step_per_class.to_csv(two_step_per_class_path, index=False)

print('Saved comparison summary to :', comparison_summary_path)
print('Saved one-step per-class to :', one_step_per_class_path)
print('Saved two-step history to   :', two_step_history_path)
print('Saved two-step per-class to :', two_step_per_class_path)


## Step 7 - Interpretation Notes

Use the following discussion points when drafting the report:

- **One-step model:** simpler end-to-end pipeline, direct detection metrics, and easier deployment for real-time use.
- **Two-step model:** more modular and easier to inspect stage by stage, but it introduces additional failure points because both detection and classification must work well together.
- **Current evidence:** the comparison is now substantially fairer because the two-step notebook reports an end-to-end IoU-based detection summary and keeps the held-out test set separate from classifier checkpoint selection.
- **Remaining limitation:** the two-step end-to-end summary is still not directly mAP-based, so the one-step YOLO result remains the stricter detector benchmark.
- **Observed class pattern:** both pipelines appear to handle `walk` better than `sit`, suggesting that `sit` is the hardest class and should be discussed explicitly in the report.


## Next Project Step

After this comparison notebook, the next major task should be one of:

- low-light robustness experiments
- condition-based analysis using lighting, viewpoint, and distance metadata
- GUI implementation for live webcam inference and fall alerting

From a marking perspective, the most valuable next step is usually condition-based analysis plus the low-light extension, because your dataset already supports that direction.